In [12]:
import shapely.geometry as sg
import glob
import re
from image_analysis_functions import extract_unique

import ee 
import geemap
import geopandas as gpd
import pandas as pd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

In [15]:
overlap_directory = './data/overlap_dates_for_roi/'
overlap_file_list = glob.glob(f'{overlap_directory}/YKD_*.shp')

def extract_roi_name(file_path: str) -> str:
    # Regex pattern: capture text that includes ROI letters, "_sub", and one or more digits
    pattern = re.compile(r".*/([A-Z]+_sub\d+)_overlap_dates\.shp$")
    match = pattern.search(file_path)
    if match:
        return match.group(1)
    else:
        return None
    
footprints = []

for f in overlap_file_list:
    gdf = gpd.read_file(f)
    roi_name = extract_roi_name(f)
    gdf['roi_name'] = roi_name
    footprints.append(gdf)

footprints = pd.concat(footprints, ignore_index=True)
footprints.to_crs(crs='EPSG:4326', inplace=True)


['./data/overlap_dates_for_roi/YKD_sub10_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub8_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub9_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub5_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub1_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub4_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub2_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub7_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub3_overlap_dates.shp', './data/overlap_dates_for_roi/YKD_sub6_overlap_dates.shp']


In [22]:
def geom_to_ee_polygon(geom):
    """
    Converts a geopandas geometry into an EE Polygon
    """
    coords = geom.exterior.coords
    coords_list = [[x, y] for x, y in coords]
    polygon = ee.Geometry.Polygon(coords_list)

    return polygon

In [44]:
def fetch_ls_angles(row: gpd.GeoSeries):
    
    asset_string = 'LANDSAT/LC08/C02/T1_TOA'
    date = row['date']
    start_date = ee.Date(date)
    end_date = start_date.advance(1, 'day')
    
    polygon = geom_to_ee_polygon(row.geometry)

    # Filter the collection by date and location.
    collection = ee.ImageCollection(asset_string) \
        .filterDate(start_date, end_date) \
        .filterBounds(polygon) \
        .select(['SAA', 'SZA'])

    # For each image in the collection get the mean, min and max of 'SAA' and 'SZA' bands
    reducer = ee.Reducer.mean() \
        .combine(ee.Reducer.minMax(), sharedInputs=True)
    
    def compute_stats(img):
        stats = img.reduceRegion(
            reducer=reducer,
            geometry=polygon,
            scale=30,
            maxPixels=1e13
        )
        return img.set(stats)
    
    collection = collection.map(compute_stats)

    collection_size = collection.size().getInfo()
    print(f'{collection_size} LS8 images in footprint')

    if collection_size > 0:
        image_list = collection.toList(collection_size)
        all_stats = []

        for i in range(collection_size):
            img = ee.Image(image_list.get(i))
            stats = img.reduceRegion(
                reducer=reducer, 
                geometry=polygon,
                scale=30,
                maxPixels=1e13
            ).getInfo()
            all_stats.append(stats)

            attrs_sun_azimuth = img.get('SUN_AZIMUTH').getInfo()
            attrs_sun_zenith = img.get('SUN_ELEVATION').getInfo()

            attrs = {'attrs_sun_azimuth': attrs_sun_azimuth,
                     'attrs_sun_zenith': attrs_sun_zenith}

        return collection_size, all_stats, attrs
    
    else: 
        return 0, None, None


In [49]:
def fetch_s2_angles(row: gpd.GeoSeries):
    
    asset_string = "COPERNICUS/S2_HARMONIZED"

    asset_string = 'LANDSAT/LC08/C02/T1_TOA'
    date = row['date']
    start_date = ee.Date(date)
    end_date = start_date.advance(1, 'day')
    
    polygon = geom_to_ee_polygon(row.geometry)

    # # Filter the collection by date and location.
    collection = ee.ImageCollection(asset_string) \
        .filterDate(start_date, end_date) \
        .filterBounds(polygon)
    
    collection_size = collection.size().getInfo()
    print(f'{collection_size} S2 images in footprint')
    if collection_size > 0:
        image_list = collection.toList(collection_size)
        all_attrs = []

        for i in range(collection_size):
            img = ee.Image(image_list.get(i))
            sun_azimuth = img.get('MEAN_SOLAR_AZIMUTH_ANGLE').getInfo()
            sun_zenith = img.get('MEAN_SOLAR_ZENITH_ANGLE').getInfo()

            attrs = {'attrs_sun_azimuth': sun_azimuth,
                     'attrs_sun_zenith': sun_zenith}
            
        return collection_size, attrs
    
    else: 
        return 0, None


In [ ]:
angle_info = []
for idx, row in footprints.iterrows():

    img_cnt, angle_stats, angle_attrs = fetch_ls_angles(row)
    new = row.copy()
    new['ls_img_cnt'] = img_cnt
    new['ls_angle_stats'] = angle_stats
    new['ls_angle_attrs'] = angle_attrs

    img_cnt, angle_attrs = fetch_s2_angles(row)
    new['s2_img_cnt'] = img_cnt
    new['s2_angle_attrs'] = angle_attrs

    angle_info.append(new)

footprints_with_angles = pd.DataFrame(angle_info)


1 LS8 images in footprint
1 S2 images in footprint


In [47]:
footprints_with_angles.head(50)

,date,int_sqkm,per_cover,geometry,roi_name,ls_img_cnt,ls_angle_stats,ls_angle_attrs
0,2018-05-23,1144.48,100.0,POLYGON ((-162.5874307457184 61.28522463931563...,YKD_sub10,1,"[{'SAA_max': 16477, 'SAA_mean': 16422.41033505...","{'attrs_sun_azimuth': 164.20285928, 'attrs_sun..."
1,2019-06-04,1144.48,100.0,POLYGON ((-162.5874307457184 61.28522463931563...,YKD_sub10,2,"[{'SAA_max': 16193, 'SAA_mean': 16137.54613509...","{'attrs_sun_azimuth': 161.90294474, 'attrs_sun..."
2,2019-06-27,1144.48,100.0,POLYGON ((-162.03631431891105 61.2383325814845...,YKD_sub10,1,"[{'SAA_max': 16225, 'SAA_mean': 16170.11459014...","{'attrs_sun_azimuth': 161.81704831, 'attrs_sun..."
3,2019-08-23,1144.48,100.0,POLYGON ((-162.5874307457184 61.28522463931563...,YKD_sub10,2,"[{'SAA_max': 16306, 'SAA_mean': 16257.74158590...","{'attrs_sun_azimuth': 163.15943803, 'attrs_sun..."
4,2020-06-13,1144.48,100.0,POLYGON ((-162.5874307457184 61.28522463931563...,YKD_sub10,1,"[{'SAA_max': 16320, 'SAA_mean': 16264.79341481...","{'attrs_sun_azimuth': 162.77004844, 'attrs_sun..."
5,2021-06-16,1144.48,100.0,POLYGON ((-162.03631431891105 61.2383325814845...,YKD_sub10,1,"[{'SAA_max': 16304, 'SAA_mean': 16248.06639897...","{'attrs_sun_azimuth': 162.57022398, 'attrs_sun..."
6,2024-06-17,1144.48,100.0,POLYGON ((-162.06838417455413 61.2916925093613...,YKD_sub10,2,"[{'SAA_max': 16048, 'SAA_mean': 15991.92652333...","{'attrs_sun_azimuth': 160.42962648, 'attrs_sun..."
7,2022-05-27,1144.48,100.0,POLYGON ((-162.5874307457184 61.28522463931563...,YKD_sub10,2,"[{'SAA_max': 16266, 'SAA_mean': 16211.02717737...","{'attrs_sun_azimuth': 162.68096796, 'attrs_sun..."
8,2022-05-02,1144.48,100.0,POLYGON ((-162.5874307457184 61.28522463931563...,YKD_sub10,1,"[{'SAA_max': 16595, 'SAA_mean': 16545.63147979...","{'attrs_sun_azimuth': 165.51951656, 'attrs_sun..."
9,2020-05-19,764.10,67.0,POLYGON ((-162.2004365213197 61.29169250936132...,YKD_sub10,1,"[{'SAA_max': 16711, 'SAA_mean': 16662.52793593...","{'attrs_sun_azimuth': 164.73787162, 'attrs_sun..."


In [56]:
# True-color composite: Landsat 8 bands B4 (red), B3 (green), and B2 (blue).
rgb_vis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
}


# For visualizing angle properties, create constant images.
ssa_angle_vis = {
    'min': 17007,
    'max': 17327,
    'palette': ['blue', 'green', 'red']
}

sza_angle_vis = {
    'min': 5248,
    'max': 5295,
    'palette': ['blue', 'green', 'red']
}


In [50]:

Map = geemap.Map(zoom=8)

# Add the Landsat‑8 mosaic layer.
Map.addLayer(mosaic, rgb_vis, 'Landsat-8 Mosaic RGB')
Map.addLayer(ssa_band, ssa_angle_vis, "SAA")
Map.addLayer(sza_band, sza_angle_vis, "SZA")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [9]:
# area_data = pd.read_csv('./data/area_data_v1.csv')
# area_data = area_data[area_data['level'] == 'toa']
# area_data = area_data[['date', 'roi', 'ls_s2_percent_diff']]

# area_data_dirty = area_data[
#     (area_data['ls_s2_percent_diff'] < -25) |
#     (area_data['ls_s2_percent_diff'] > 25)
# ]

# area_data_clean = area_data[
#     (area_data['ls_s2_percent_diff'] > -5) &
#     (area_data['ls_s2_percent_diff'] < 5)
# ]

# area_data_dirty.head(20)
# area_data_clean.head(20)

,date,roi,ls_s2_percent_diff
56,2017-07-12,MRD_sub1,0.408118
57,2022-08-11,MRD_sub1,-0.092062
58,2023-07-29,MRD_sub1,1.765872
59,2016-06-07,MRD_sub1,0.892717
60,2020-06-02,MRD_sub1,-1.562868
61,2024-06-15,MRD_sub1,-0.294320
62,2020-06-04,MRD_sub1,-2.406787
63,2019-07-27,MRD_sub1,0.584560
64,2021-06-14,MRD_sub1,-0.620002
65,2023-06-29,MRD_sub1,0.294484
